# Curious Minds Club Chatbot

A topic-locked club assistant. Conversation state is persisted in Supabase PostgreSQL through LangGraph’s PostgresSaver. Run the cells from top to bottom in Google Colab; add OPENAI_API_KEY and SUPABASE_DB_URL to Colab Secrets first.

In [ ]:
!pip -q install langgraph langchain langchain-openai langgraph-checkpoint-postgres "psycopg[binary]" gradio

In [ ]:
import os
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
SUPABASE_DB_URL = userdata.get("SUPABASE_DB_URL")

if not os.environ["OPENAI_API_KEY"] or not SUPABASE_DB_URL:
    raise ValueError("Add OPENAI_API_KEY and SUPABASE_DB_URL to Google Colab Secrets.")

## Build the topic-locked graph

The system message is created for each turn from the saved state topic. It is passed to the model but never added to the persisted message history.

In [ ]:
from typing import Annotated, TypedDict

from IPython.display import Image, display
from langchain.chat_models import init_chat_model
from langchain_core.messages import AnyMessage, SystemMessage
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages

MODEL_NAME = "openai:gpt-4o-mini"
TOPICS = ["Astronomy", "Cooking", "Football", "World Geography", "Python Programming"]
model = init_chat_model(MODEL_NAME, temperature=0)

class ChatState(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]
    topic: str

def chatbot(state: ChatState):
    topic = state["topic"]
    system_msg = SystemMessage(content=(
        f"You are a helpful assistant for the {topic} club at Curious Minds. "
        f"Answer only questions strictly related to {topic}. Politely refuse any off-topic question, "
        f"explicitly mention that we are currently in the {topic} club, and suggest switching to another "
        f"club from the dropdown ({', '.join(TOPICS)}). If the user asks a follow-up question about a "
        f"previous topic that appears earlier in the conversation history, you must refuse it and remind "
        f"them that we are now in the {topic} club."
    ))
    reply = model.invoke([system_msg] + state["messages"])
    return {"messages": [reply]}

builder = StateGraph(ChatState)
builder.add_node("chatbot", chatbot)
builder.add_edge(START, "chatbot")
builder.add_edge("chatbot", END)

## Connect the Supabase checkpointer

In [ ]:
import psycopg
from psycopg.rows import dict_row
from langgraph.checkpoint.postgres import PostgresSaver

conn = psycopg.connect(
    userdata.get("SUPABASE_DB_URL"),
    autocommit=True,
    row_factory=dict_row,
    prepare_threshold=0,
)
checkpointer = PostgresSaver(conn)
checkpointer.setup()
graph = builder.compile(checkpointer=checkpointer)

## Graph diagram

In [ ]:
display(Image(graph.get_graph().draw_mermaid_png()))

## Standalone graph test

This exercises the graph directly before launching Gradio. It uses a persistent test thread named test-1.

In [ ]:
from langchain_core.messages import HumanMessage

config = {"configurable": {"thread_id": "test-1"}}

def ask(text, topic):
    result = graph.invoke({"messages": [HumanMessage(text)], "topic": topic}, config)
    print(f"[{topic}] {text}\n -> {result['messages'][-1].content}\n")

ask("How do I make pancakes?", "Cooking")
ask("Who won the 2022 World Cup?", "Cooking")
ask("Who won the 2022 World Cup?", "Football")
ask("And how long do I fry each pancake?", "Football")
graph.get_state(config).values

## Gradio chat interface

Saved conversation options are discovered from the Supabase checkpoints table. The dropdown shows a readable topic and first user message while retaining the thread ID as its option value.

In [ ]:
import uuid
import gradio as gr
from langchain_core.messages import AIMessage, HumanMessage

def thread_config(thread_id):
    return {"configurable": {"thread_id": thread_id}}

def to_chat_messages(messages):
    chat = []
    for message in messages:
        if isinstance(message, HumanMessage):
            role = "user"
        elif isinstance(message, AIMessage):
            role = "assistant"
        else:
            continue
        content = message.content
        chat.append({"role": role, "content": content if isinstance(content, str) else str(content)})
    return chat

def list_saved_conversations():
    # PostgresSaver.setup() creates this table; query persisted thread IDs directly.
    with conn.cursor() as cursor:
        cursor.execute("SELECT DISTINCT thread_id FROM checkpoints ORDER BY thread_id")
        thread_ids = [row["thread_id"] for row in cursor.fetchall()]

    choices = []
    for thread_id in thread_ids:
        values = graph.get_state(thread_config(thread_id)).values or {}
        messages = values.get("messages", [])
        topic = values.get("topic", "Unknown club")
        first_user = next((m.content for m in messages if isinstance(m, HumanMessage)), "Empty conversation")
        if not isinstance(first_user, str):
            first_user = str(first_user)
        preview = first_user[:30] + ("…" if len(first_user) > 30 else "")
        choices.append((f"{topic} — {preview}", thread_id))
    return choices

def send_message(text, topic, thread_id, history):
    if not text or not text.strip():
        return history, "", thread_id, gr.update(choices=list_saved_conversations())
    result = graph.invoke(
        {"messages": [HumanMessage(content=text)], "topic": topic},
        thread_config(thread_id),
    )
    updated_history = to_chat_messages(result["messages"])
    return updated_history, "", thread_id, gr.update(choices=list_saved_conversations(), value=None)

def load_conversation(thread_id):
    if not thread_id:
        return [], TOPICS[0], str(uuid.uuid4())
    values = graph.get_state(thread_config(thread_id)).values or {}
    topic = values.get("topic", TOPICS[0])
    if topic not in TOPICS:
        topic = TOPICS[0]
    return to_chat_messages(values.get("messages", [])), topic, thread_id

def new_conversation():
    return [], str(uuid.uuid4()), None

with gr.Blocks(title="Curious Minds Club Chatbot", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# Curious Minds Club Chatbot\nChoose a club and ask a question. Your conversation is saved automatically.")
    with gr.Row():
        topic_dropdown = gr.Dropdown(choices=TOPICS, value=TOPICS[0], label="Club / Topic")
        saved_dropdown = gr.Dropdown(choices=list_saved_conversations(), label="Saved Conversations", value=None)
        new_button = gr.Button("+ New conversation", variant="secondary")
    chat = gr.Chatbot(type="messages", label="Chat", height=480)
    message_box = gr.Textbox(placeholder="Type your message and press Enter", label="Message", lines=2)
    active_thread_id = gr.State(str(uuid.uuid4()))

    message_box.submit(
        send_message,
        inputs=[message_box, topic_dropdown, active_thread_id, chat],
        outputs=[chat, message_box, active_thread_id, saved_dropdown],
    )
    saved_dropdown.change(
        load_conversation,
        inputs=[saved_dropdown],
        outputs=[chat, topic_dropdown, active_thread_id],
    )
    new_button.click(
        new_conversation,
        inputs=[],
        outputs=[chat, active_thread_id, saved_dropdown],
    )

demo.launch(share=True)